# 05 Capstone: release decision

## Learning objectives

- run a controlled text/vector/hybrid/reranked ablation on one ordered dataset;
- inspect absolute gates and regression policy separately;
- choose `adopt`, `reject`, or `inconclusive` from evidence;
- describe how the notebook graduates into the `rag-app` or `agent-app` template.


In [ ]:
# Notebook preflight — configuration only; this cell makes no cloud request.
import importlib.util
import sys
from pathlib import Path

setup_path = next(
    path
    for parent in (Path.cwd(), *Path.cwd().parents)
    for path in (
        parent / "notebook_setup.py",
        parent / "examples" / "agentic-ops-rag" / "notebook_setup.py",
    )
    if path.is_file()
)
spec = importlib.util.spec_from_file_location("agentic_ops_rag_setup", setup_path)
setup = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = setup
spec.loader.exec_module(setup)
course_root = setup.find_course_root(setup_path.parent)
session = setup.prepare_notebook_environment(course_root)
session.safe_summary()

## The capstone contract

Hold the synthetic corpus, cases, prompt behavior, access scope, and answer
policy constant. Change one retrieval decision at a time. A fast configuration
that leaks a tenant or executes an action is ineligible; a safe configuration
that loses answerable coverage is not rescued by a high conditional score.


In [ ]:
from agentic_ops_rag import RetrievalMode
from agentic_ops_rag.evaluation import (
    benchmark,
    comparison_record,
    is_release_eligible,
    load_cases,
    release_gate,
)

pipeline = session.offline_pipeline()
cases = load_cases(course_root / "data" / "evaluation_cases.jsonl")
configurations = {
    "A_text": (RetrievalMode.TEXT, False),
    "B_vector": (RetrievalMode.VECTOR, False),
    "C_hybrid": (RetrievalMode.HYBRID, False),
    "D_hybrid_reranked": (RetrievalMode.HYBRID, True),
}
reports = {
    name: benchmark(pipeline, cases, mode=mode, semantic_rerank=rerank)
    for name, (mode, rerank) in configurations.items()
}
reports

Do not select the configuration with the largest provider score: the score
semantics differ. Compare retrieval recall and MRR, abstention coverage,
citation integrity, tenant isolation, action approval, latency, and cost
coverage. Here cost coverage is zero because the offline fixture has no
provider invoice; unknown is more honest than a fabricated number.


In [ ]:
absolute_gates = {name: release_gate(metrics) for name, metrics in reports.items()}
absolute_gate_summary = {
    name: {
        "passed": gate.passed,
        "failures": [failure.reason for failure in gate.failures],
    }
    for name, gate in absolute_gates.items()
}
absolute_gate_summary

## Baseline versus one controlled change

An absolute gate answers “is this configuration eligible?” A regression gate
also asks whether its gain justifies degradation from the current baseline.
Here the hybrid change genuinely improves recall and MRR on the fixed cases,
so the vector-to-hybrid comparison should come back `adopt`. Note the latency
rule's role: the offline latencies are labelled `simulated_offline_fixture`
and only sketch a trade-off shape, so the policy grants them a wide regression
allowance and a real quality gain is not vetoed by an invented number. In a
connected deployment the latency budget comes from measured traces, and a
change that regressed it would be rejected on real evidence.


In [ ]:
comparison = comparison_record(
    reports["B_vector"],
    reports["C_hybrid"],
    baseline_configuration="B_vector",
    change_configuration="C_hybrid",
)
comparison.model_dump(mode="json")

In [ ]:
# YOUR TURN — TODO: make an evidence-backed lifecycle decision.
if comparison.failures:
    learner_decision = "reject"
elif not reports["C_hybrid"]:
    learner_decision = "inconclusive"
else:
    learner_decision = "adopt"
learner_decision

In [ ]:
# CHECK YOUR WORK
assert learner_decision in {"adopt", "reject", "inconclusive"}
if learner_decision == "adopt":
    assert not comparison.failures
elif comparison.failures:
    assert learner_decision == "reject"
"The decision follows the recorded gate evidence."

In [ ]:
# Reference solution
reference_decision = comparison.decision
assert learner_decision == reference_decision
decision_evidence = {
    "baseline_configuration": comparison.baseline_configuration,
    "change_configuration": comparison.change_configuration,
    "decision": reference_decision,
    "failures": [failure.model_dump(mode="json") for failure in comparison.failures],
    "measurement_source": "simulated_offline_fixture",
}
decision_evidence

## A forged decision does not survive the gate

The decision field is evidence, not authority. Record a genuinely rejected
comparison — moving from hybrid back to text loses recall beyond the
regression allowance — then forge its decision to `adopt` and watch
`is_release_eligible` refuse it anyway: eligibility recomputes the gate from
the recorded metrics and requires the recomputed result to agree with the
record.


In [ ]:
downgrade = comparison_record(
    reports["C_hybrid"],
    reports["A_text"],
    baseline_configuration="C_hybrid",
    change_configuration="A_text",
)
assert downgrade.decision == "reject"
forged = downgrade.model_copy(update={"decision": "adopt", "failures": ()})
forged_eligible = is_release_eligible(
    "A_text",
    absolute_gate=release_gate(reports["A_text"]),
    baseline_metrics=reports["C_hybrid"],
    comparison=forged,
    source_state="clean",
)
assert not forged_eligible
print("honest decision:", downgrade.decision)
print("recorded failures:", [failure.metric for failure in downgrade.failures])
print("forged decision:  ", forged.decision)
print("is_release_eligible(forged):", forged_eligible)
print("editing the record does not edit the evidence: the gate is recomputed.")

## Immutable release evidence

Only an eligible choice becomes an application release. The release ties code,
model, prompt, retrieval, evaluation, and environment together. A prompt alias,
mutable index name, or notebook output is not sufficient release lineage.

Eligibility is a conjunction of independently checkable preconditions, so print
them as a checklist rather than a single boolean: a dirty working tree and a
rejected comparison both make `is_release_eligible` return `False`, and an
operator has to be able to tell those situations apart.


In [ ]:
import subprocess

from aai_core import __version__ as aai_core_version
from aai_core.deployment import ApplicationRelease

commit_result = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=session.repository_root,
    capture_output=True,
    text=True,
    check=False,
)
state_result = subprocess.run(
    ["git", "status", "--porcelain", "--untracked-files=no"],
    cwd=session.repository_root,
    capture_output=True,
    text=True,
    check=False,
)
source_commit = commit_result.stdout.strip()
git_provenance_available = (
    commit_result.returncode == 0
    and state_result.returncode == 0
    and bool(source_commit)
)
source_state = (
    "clean" if git_provenance_available and not state_result.stdout.strip() else "dirty"
)
print("live source_state:", source_state)

In [ ]:
from agentic_ops_rag import ComparisonRecord

selected_name = comparison.change_configuration
selected_gate = absolute_gates[selected_name]
trusted_baseline = reports[comparison.baseline_configuration]
recomputed_gate = release_gate(
    dict(comparison.change),
    baseline_metrics=dict(trusted_baseline),
)
gate_metrics = dict(selected_gate.metrics)
eligibility_checklist = {
    "comparison_is_a_strict_record": type(comparison) is ComparisonRecord,
    "source_tree_is_clean": source_state == "clean",
    "absolute_gate_passed": selected_gate.passed,
    "recorded_baseline_matches_trusted_baseline": (
        dict(comparison.baseline) == dict(trusted_baseline)
    ),
    "selected_configuration_is_the_recorded_change": (
        comparison.change_configuration == selected_name
    ),
    "recorded_change_and_result_match_gate_metrics": (
        dict(comparison.change) == gate_metrics
        and dict(comparison.result) == gate_metrics
    ),
    "recomputed_gate_agrees_with_the_record": (
        dict(recomputed_gate.metrics) == gate_metrics
        and recomputed_gate.failures == comparison.failures
        and recomputed_gate.passed
    ),
    "decision_is_adopt_with_no_recorded_failures": (
        comparison.decision == "adopt" and not comparison.failures
    ),
}
release_eligible = is_release_eligible(
    selected_name,
    absolute_gate=selected_gate,
    baseline_metrics=reports[comparison.baseline_configuration],
    comparison=comparison,
    source_state=source_state,
)
for check_name, check_passed in eligibility_checklist.items():
    print(f"[{'PASS' if check_passed else 'FAIL'}] {check_name}")
print("is_release_eligible:", release_eligible)
assert release_eligible == all(eligibility_checklist.values())

## The digest demonstration and the honest checklist

This notebook also runs in automated verification, where the working tree
legitimately contains the notebook execution itself — so `source_tree_is_clean`
can honestly be `FAIL` while every evidence check passes. The checklist above
always reports the live tree. To keep the mechanism teachable in both worlds,
the cell below first re-verifies that the recorded comparison is adopt-grade
under an explicit `demonstration_source_state`, then builds the release record
and prints its digest. The record itself carries both values: the
demonstration state it was built under and the live state observed at render
time. A publishable release is only ever cut when the live checklist passes
end-to-end from a committed tree.


In [ ]:
if release_eligible:
    provenance_note = (
        "live tree is clean: this digest is real, publishable release evidence"
    )
else:
    provenance_note = (
        "live tree is dirty (normal while editing or during automated "
        "notebook execution): the digest below demonstrates the mechanism "
        "under an explicit demonstration source_state and must not ship"
    )
demonstration_source_state = "clean"
adopt_grade_evidence = is_release_eligible(
    selected_name,
    absolute_gate=selected_gate,
    baseline_metrics=reports[comparison.baseline_configuration],
    comparison=comparison,
    source_state=demonstration_source_state,
)
assert adopt_grade_evidence, "never demonstrate a digest from rejected evidence"
release = ApplicationRelease(
    application="operations-rag-assistant",
    release="workshop-hybrid-v1",
    source_commit=source_commit if git_provenance_available else "unavailable",
    core_sdk_version=aai_core_version,
    model={"logical_name": "operations-chat", "version": "configured"},
    prompt={"name": "operations-system", "version": 1},
    retrieval={
        "logical_name": "operations-knowledge",
        "mode": "hybrid",
        "chunking_profile": "markdown-structural-v1",
        "embedding_profile": "operations-embedding-v1",
    },
    evaluation={
        "dataset": "synthetic-operations-regression-v1",
        "gate_passed": selected_gate.passed,
        "comparison": comparison.model_dump(mode="json"),
        "metrics": reports[selected_name],
        "source_state": demonstration_source_state,
        "source_state_at_render": source_state,
    },
    environment="dev",
)
print(provenance_note)
print("decision:", comparison.decision)
print("release digest:", release.digest)

The digest is a canonical hash over the whole record, which is what makes the
release immutable in practice: any change to any field — a different release
name, one edited metric, a swapped comparison — produces a different digest,
so evidence cannot drift silently after the fact.


In [ ]:
release_variant = release.model_copy(update={"release": "workshop-hybrid-v2"})
assert release.digest != release_variant.digest
print("workshop-hybrid-v1 digest:", release.digest)
print("workshop-hybrid-v2 digest:", release_variant.digest)
print("one changed field, a different digest: evidence cannot drift silently.")

## Graduation into the stack

- Use `rag-app` when retrieval plus generation is the product boundary. It
  packages code under `src/`, builds chunks in a job, pins prompts, evaluates
  with MLflow, and deploys a governed bundle.
- Use `agent-app` when tools and actions are required. Its primary HTTP path is
  MLflow Agent Server on Databricks Apps. It keeps typed async tools, timeouts,
  exact trajectory checks, and optional durable LangGraph interrupts.
- Keep Azure AI Search or Databricks AI Search behind
  `operations-knowledge`. Provision indexes and roles through the external
  platform process, not this notebook or CI.
- Load test after CI/CD and before production. Small offline p95 samples are
  teaching evidence, never an SLA claim.


In [ ]:
RUN_CONNECTED = False
connected_capstone = None
if RUN_CONNECTED:
    resources = session.connected_components(allow_network=True)
    connected_capstone = {
        "model_provider": resources["model"].provider,
        "retrieval_provider": resources["retriever"].provider,
        "next_step": "run the fixed MLflow evaluation before any deployment",
    }
connected_capstone

## Knowledge check

Answer from the evidence you produced, not from memory:

1. Why can an absolute gate pass while a regression decision rejects the change?
2. Which release fields change when chunking or embeddings change?
3. When should this project graduate to rag-app versus agent-app?

<details>
<summary>How to use this check</summary>

If an answer cannot point to a row, trace, contract, or failed check from this
lesson, revisit the exercise before moving on.

</details>


## Recap

You completed the full lifecycle: baseline, controlled change, result, an
`adopt` decision earned on row-level evidence, and a digest-sealed release
record that refuses forged decisions. The result stays reproducible offline,
while every real model, judge, search, trace, and deployment operation is
explicit, keyless, and governed by the surrounding platform. Lesson 06 asks the
question every one of those numbers deserves: how sure are we?
